In [ ]:
%pip install -q numpy matplotlib

## Scenario

This notebook covers two deep learning architecture families from the Week 1 Friday lecture:

- **Stage 1 -- CNN Image Classifier:** Design a convolutional neural network that classifies images, understand data augmentation, and optionally add residual connections.
- **Stage 2 -- Encoder-Decoder Sequence Translation:** Combine encoder and decoder LSTMs into a seq2seq model that learns to reverse digit strings, demonstrating the encoder-decoder pattern used in machine translation.

The companion notebook `dl_rnn_lstm_gru.ipynb` (Monday) covers the RNN, LSTM, and GRU architectures in detail. A brief LSTM primer is included below so you can follow the encoder-decoder architecture without the full Monday material.

## STAGE 1 -- CNNs for Image Data (55 min)

## STEP 1.1 -- Why CNNs? From MLPs to Convolutions (8 min)

## STEP 1.2 -- Exploring Image Data (8 min)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck"]

# In practice, load a dataset like CIFAR-10 from your framework of choice.
# Images are 32x32 pixels with 3 color channels (RGB).
# Standard normalization uses per-channel means and standard deviations:
#   means = (0.4914, 0.4822, 0.4465)
#   stds  = (0.2470, 0.2435, 0.2616)
#
# For this conceptual walkthrough, we simulate an image batch:
SUBSET_SIZE = 5000
TEST_SUBSET = 1000

# Simulated image batch for shape demonstration
sample_images = np.random.rand(10, 32, 32, 3)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(sample_images[i])
    ax.set_title(CLASSES[i % len(CLASSES)])
    ax.axis("off")
plt.suptitle("Sample Images (32x32x3)")
plt.tight_layout()
plt.show()

print(f"Image shape:        (3, 32, 32)  [channels, height, width]")
print(f"Training samples:   {SUBSET_SIZE}")
print(f"Test samples:       {TEST_SUBSET}")
print(f"Number of classes:  {len(CLASSES)}")

## STEP 1.3 -- Design a Basic CNN (15 min)

## STEP 1.4 -- CNN Training Concepts (10 min)

## STEP 1.5 -- Data Augmentation (8 min)

## STEP 1.6 -- Residual Connections (Stretch) (6 min)

---
## LSTM Primer -- Conceptual Bridge to Encoder-Decoder

The encoder-decoder architecture (Stage 2 below) uses LSTM cells internally. The full RNN/LSTM/GRU treatment is in the Monday notebook (`dl_rnn_lstm_gru.ipynb`). Here is what you need to follow the encoder-decoder pattern:

**What an LSTM does:**
- An LSTM processes a sequence one step at a time, like reading a sentence word by word.
- At each step, it takes an input token and its previous internal state, and produces an updated state.
- It maintains two state vectors:
  - **Hidden state** `h_t` -- a working summary of the sequence so far.
  - **Cell state** `c_t` -- a long-term memory highway where information can persist across many steps.
- After processing the entire sequence, the final states `(h_T, c_T)` summarize the input.

**Why LSTM instead of a simpler recurrence:**
- A basic recurrent cell multiplies gradients by the same weight matrix at every time step, causing gradients to vanish on long sequences.
- LSTMs use **gates** (learned, element-wise controls) that let gradients flow through the cell state via addition rather than repeated multiplication.
- The result: LSTMs can capture dependencies across dozens or hundreds of time steps.

**How it connects to encoder-decoder:**
- The **encoder** is an LSTM that reads the input sequence and outputs `(h_T, c_T)` as a context vector.
- The **decoder** is another LSTM that receives the context vector as its initial state and generates the output sequence one token at a time.

Monday's notebook covers the gate equations, vanishing gradients, and the GRU alternative in full detail.

---
## STAGE 2 -- Encoder-Decoder Architectures (45 min)

## STEP 2.1 -- The Encoder-Decoder Idea (8 min)

## STEP 2.2 -- Generate Sequence Reversal Data (5 min)

In [ ]:
import numpy as np

PAD_TOKEN = 0
SOS_TOKEN = 11
EOS_TOKEN = 12
NUM_TOKENS = 13  # 0=PAD, 1-10=digits, 11=SOS, 12=EOS
MAX_LEN = 6

def generate_reversal_data(n_samples, max_len=MAX_LEN, seed=42):
    rng = np.random.RandomState(seed)
    encoder_inputs = []
    decoder_inputs = []
    decoder_targets = []

    for _ in range(n_samples):
        length = rng.randint(3, max_len + 1)
        seq = rng.randint(1, 11, size=length).tolist()
        reversed_seq = seq[::-1]

        enc_in = seq + [PAD_TOKEN] * (max_len - length)
        dec_in = [SOS_TOKEN] + reversed_seq + [PAD_TOKEN] * (max_len - length)
        dec_tgt = reversed_seq + [EOS_TOKEN] + [PAD_TOKEN] * (max_len - length)

        encoder_inputs.append(enc_in)
        decoder_inputs.append(dec_in)
        decoder_targets.append(dec_tgt)

    return (np.array(encoder_inputs),
            np.array(decoder_inputs),
            np.array(decoder_targets))

enc_train, dec_in_train, dec_tgt_train = generate_reversal_data(3000, seed=42)
enc_test, dec_in_test, dec_tgt_test = generate_reversal_data(500, seed=99)

print(f"Encoder input shape:  {enc_train.shape}")
print(f"Decoder input shape:  {dec_in_train.shape}")
print(f"Decoder target shape: {dec_tgt_train.shape}")
print(f"\nExample:")
print(f"  Encoder input:  {enc_train[0].tolist()}")
print(f"  Decoder input:  {dec_in_train[0].tolist()}")
print(f"  Decoder target: {dec_tgt_train[0].tolist()}")

## STEP 2.3 -- Design the Encoder (7 min)

## STEP 2.4 -- Design the Decoder (7 min)

## STEP 2.5 -- Assemble the Seq2Seq Model (5 min)

## STEP 2.6 -- Seq2Seq Training Concepts (8 min)

## STEP 2.7 -- Qualitative Evaluation (5 min)